# 05 — Raw data and DataFrame workflows

The object API is convenient for exploration, while `data` provides read-only access to the original machine-readable catalogue structure for inspection and downstream analysis.

In [ ]:
import json

import pandas as pd
import xeo

## Read raw catalogue records

`catalogue.data` is JSON-serializable and follows the upstream catalogue structure. Treat it as read-only catalogue input; use it to inspect records or pass metadata to downstream workflows.

In [ ]:
raw_record = xeo.catalogue.data["instruments"]["MSI_S2A"]
print(raw_record.keys())
print(json.dumps({"id": raw_record["id"], "name": raw_record["name"]}, indent=2))

## Build a catalogue summary DataFrame

A compact DataFrame is useful for filtering, sorting, and counting instruments without flattening the larger extension and SRF data.

In [ ]:
summary = pd.DataFrame(
    [
        {
            "id": instrument.id,
            "name": instrument.name,
            "acronym": instrument.acronym,
            "type": instrument.type,
            "platform_type": instrument.platform_type,
            "platform": ", ".join(instrument.platform),
            "status": instrument.status,
            "availability": instrument.availability,
            "has_bands": instrument.has_bands,
            "has_srf": instrument.has_srf,
        }
        for instrument in xeo.instruments.values()
    ]
).set_index("id")
summary.head()

## Filter and aggregate

In [ ]:
operational_satellites = summary.loc[
    (summary["platform_type"] == "satellite")
    & (summary["status"] == "operational")
].sort_values(["type", "name"])
operational_satellites

In [ ]:
summary.groupby(["platform_type", "status"]).size().rename("instrument_count")